In [103]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.metrics import matthews_corrcoef, precision_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split, StratifiedKFold, TimeSeriesSplit
from xgboost import XGBClassifier
import pickle
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
from hmmlearn.hmm import GaussianHMM
import re
import itertools
import time
import warnings
warnings.filterwarnings("ignore", module="joblib")
import databento as db
import exchange_calendars as xcals

# Read the DBN file into a DBNStore object
dbn_store = db.DBNStore.from_file('qqq_1m.dbn')
# Convert the data to a pandas DataFrame for analysis
df = dbn_store.to_df()
df_main = df.reset_index()[['symbol', 'ts_event', 'close', 'open', 'high', 'low', 'volume']].copy()
df_main['datetime_est'] = (df_main['ts_event'].dt.tz_convert('America/New_York'))
df_main

,symbol,ts_event,close,open,high,low,volume,datetime_est
0,QQQ,2018-05-01 08:07:00+00:00,160.90,160.90,160.90,160.90,100,2018-05-01 04:07:00-04:00
1,QQQ,2018-05-01 08:14:00+00:00,160.79,160.80,160.80,160.79,300,2018-05-01 04:14:00-04:00
2,QQQ,2018-05-01 08:16:00+00:00,160.89,160.89,160.89,160.89,51,2018-05-01 04:16:00-04:00
3,QQQ,2018-05-01 08:20:00+00:00,160.88,160.88,160.88,160.88,7,2018-05-01 04:20:00-04:00
4,QQQ,2018-05-01 08:21:00+00:00,160.93,160.93,160.93,160.93,19,2018-05-01 04:21:00-04:00
...,...,...,...,...,...,...,...,...
1517292,QQQ,2025-12-19 23:51:00+00:00,618.38,618.40,618.40,618.38,157,2025-12-19 18:51:00-05:00
1517293,QQQ,2025-12-19 23:55:00+00:00,618.36,618.36,618.36,618.36,297,2025-12-19 18:55:00-05:00
1517294,QQQ,2025-12-19 23:57:00+00:00,618.46,618.46,618.46,618.46,20,2025-12-19 18:57:00-05:00
1517295,QQQ,2025-12-19 23:58:00+00:00,618.48,618.46,618.48,618.46,1007,2025-12-19 18:58:00-05:00


# 1. Session Structure & Market Phases

In [99]:
def close_times():

    # NYSE calendar
    cal = xcals.get_calendar("XNYS")

    # Build schedule for the date range you care about
    #start = "2018-01-01"
    #end = "2030-01-01"
    sched = cal.schedule.loc[:, ["open", "close"]].copy()

    # Convert to America/New_York
    sched["open_et"]  = sched["open"].dt.tz_convert("America/New_York")
    sched["close_et"] = sched["close"].dt.tz_convert("America/New_York")

    # Indicators
    #sched["is_trading_day"] = True
    sched["is_early_close"] = sched["close_et"].dt.time < pd.Timestamp("16:00", tz="America/New_York").time()

    # If you want a per-day close time (minutes since midnight ET)
    sched["session_duration"] = (sched["close_et"].dt.hour * 60 + sched["close_et"].dt.minute) - 9.5 * 60

    # Join to your intraday df by session date
    # assumes df has a Date column that is the NYSE session date (ET)
    sched_out = sched.reset_index().rename(columns={"index": "Date"})
    close_times_df = sched_out
    
    return close_times_df[['Date', 'close_et', 'is_early_close', 'session_duration']]

df_close_times = close_times()

def add_intraday_labels(df: pd.DataFrame, dt_col: str = "datetime_est") -> pd.DataFrame:
    
    out = df.copy()

    # Ensure datetime
    out[dt_col] = pd.to_datetime(out[dt_col], errors="coerce")
    if out[dt_col].isna().any():
        bad = out[dt_col].isna().sum()
        raise ValueError(f"{bad} rows in {dt_col} could not be parsed to datetime.")

    # Extract time-of-day in minutes since midnight (ET)
    tod_minutes = out[dt_col].dt.hour * 60 + out[dt_col].dt.minute
    out["_tod_minutes"] = tod_minutes

    # Open time (09:30 ET) in minutes
    premarket_min = 7 * 60  # 420
    open_min = 9 * 60 + 30  # 570
    close_min = 16 * 60 - 1   # 960

    # Minutes since open (can be negative pre-market, positive post-open)
    out["minutes_since_open"] = out["_tod_minutes"] - open_min

    # Column 1: simple session label
    out["session_simple"] = np.select(
        [
            out["_tod_minutes"] < premarket_min,
            (out["_tod_minutes"] >= premarket_min) & (out["_tod_minutes"] < open_min),
            (out["_tod_minutes"] >= open_min) & (out["_tod_minutes"] <= close_min),
            out["_tod_minutes"] > close_min,
        ],
        ["overnight", "pre_market", "open_market", "post_market"],
        default=np.nan
    )

    # Column 2: detailed session label (your buckets)
    out["session_detail"] = np.select(
        [
            # Pre-market buckets
            (out["_tod_minutes"] < 7 *60),
            (out["_tod_minutes"] >= 7*60) & (out["_tod_minutes"] < 9*60),
            (out["_tod_minutes"] >= 9*60) & (out["_tod_minutes"] < open_min),

            # Open market buckets
            (out["_tod_minutes"] >= open_min) & (out["_tod_minutes"] < 9*60+45),
            (out["_tod_minutes"] >= 9*60+45) & (out["_tod_minutes"] < 10*60),
            (out["_tod_minutes"] >= 10*60) & (out["_tod_minutes"] < 12*60),
            (out["_tod_minutes"] >= 12*60) & (out["_tod_minutes"] < 14*60),
            (out["_tod_minutes"] >= 14*60) & (out["_tod_minutes"] < 15*60+30),
            (out["_tod_minutes"] >= 15*60+30) & (out["_tod_minutes"] < 15*60+45),
            (out["_tod_minutes"] >= 15*60+45) & (out["_tod_minutes"] <= close_min),

            # Post-market buckets
            (out["_tod_minutes"] > close_min) & (out["_tod_minutes"] < 16*60+15),
            (out["_tod_minutes"] >= 16*60+15) & (out["_tod_minutes"] < 17*60),
            (out["_tod_minutes"] >= 17*60) & (out["_tod_minutes"] <= 20*60),
        ],
        [
            "overnight",
            "early_pre_market",
            "late_pre_market",
            "early_open",
            "late_open",
            "morning",
            "midday",
            "late_day",
            "early_close",
            "late_close",
            "early_post_market",
            "late_post_market",
            "post_market",
        ],
        default="other"
    )

    # Cleanup
    out = out.drop(columns=["_tod_minutes", "_detail_simple_check"], errors="ignore")
    return out

#df_intraday_labels[df_intraday_labels['minutes_since_open'] == 390]
#Shortest minutes_since_open = -330 largest is 629. 0 = 930am, 389 = 4:00pm
df_intraday_labels = add_intraday_labels(df_main)
df_intraday_labels['Date'] = pd.to_datetime(df_intraday_labels['datetime_est']).dt.strftime('%Y-%m-%d')

In [100]:
# Merge close times with intraday data
df_intraday_labels["Date"] = pd.to_datetime(df_intraday_labels["Date"]).dt.date
df_close_times["Date"] = pd.to_datetime(df_close_times["Date"]).dt.date
df_out = df_intraday_labels.merge(df_close_times, on="Date", how="left")

df_out


,symbol,ts_event,close,open,high,low,volume,datetime_est,minutes_since_open,session_simple,session_detail,Date,close_et,is_early_close,session_duration
0,QQQ,2018-05-01 08:07:00+00:00,160.90,160.90,160.90,160.90,100,2018-05-01 04:07:00-04:00,-323,overnight,overnight,2018-05-01,2018-05-01 16:00:00-04:00,False,390.0
1,QQQ,2018-05-01 08:14:00+00:00,160.79,160.80,160.80,160.79,300,2018-05-01 04:14:00-04:00,-316,overnight,overnight,2018-05-01,2018-05-01 16:00:00-04:00,False,390.0
2,QQQ,2018-05-01 08:16:00+00:00,160.89,160.89,160.89,160.89,51,2018-05-01 04:16:00-04:00,-314,overnight,overnight,2018-05-01,2018-05-01 16:00:00-04:00,False,390.0
3,QQQ,2018-05-01 08:20:00+00:00,160.88,160.88,160.88,160.88,7,2018-05-01 04:20:00-04:00,-310,overnight,overnight,2018-05-01,2018-05-01 16:00:00-04:00,False,390.0
4,QQQ,2018-05-01 08:21:00+00:00,160.93,160.93,160.93,160.93,19,2018-05-01 04:21:00-04:00,-309,overnight,overnight,2018-05-01,2018-05-01 16:00:00-04:00,False,390.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1517292,QQQ,2025-12-19 23:51:00+00:00,618.38,618.40,618.40,618.38,157,2025-12-19 18:51:00-05:00,561,post_market,post_market,2025-12-19,2025-12-19 16:00:00-05:00,False,390.0
1517293,QQQ,2025-12-19 23:55:00+00:00,618.36,618.36,618.36,618.36,297,2025-12-19 18:55:00-05:00,565,post_market,post_market,2025-12-19,2025-12-19 16:00:00-05:00,False,390.0
1517294,QQQ,2025-12-19 23:57:00+00:00,618.46,618.46,618.46,618.46,20,2025-12-19 18:57:00-05:00,567,post_market,post_market,2025-12-19,2025-12-19 16:00:00-05:00,False,390.0
1517295,QQQ,2025-12-19 23:58:00+00:00,618.48,618.46,618.48,618.46,1007,2025-12-19 18:58:00-05:00,568,post_market,post_market,2025-12-19,2025-12-19 16:00:00-05:00,False,390.0


In [107]:
# Leo to test data gaps between 'minutes_since_open 0:389